# Notebook 04 — Stage G Messy Data Storytelling & Robust Prediction

This notebook starts the next phase **without overloading Notebook 03**.

Focus:
- Missingness mechanisms (not just missing rate)
- Extreme-value stories (retain signal, do not over-clean)
- Stage-wise model complexity additions and comparison
- Permutation diagnostics (label-permutation collapse + feature permutation importance)

Design principle:
- Keep a raw layer and an observed/messy layer side by side
- Never silently drop rare/extreme events before interpretation

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

np.random.seed(42)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
TABLE_DIR = PROJECT_ROOT / 'Results' / 'tables' / 'notebook04_stage_g'
FIG_DIR = PROJECT_ROOT / 'Results' / 'figures' / 'notebook04_stage_g'
REPORT_DIR = PROJECT_ROOT / 'Results' / 'reports' / 'notebook04_stage_g'
META_DIR = PROJECT_ROOT / 'Data' / 'metadata'
for d in [TABLE_DIR, FIG_DIR, REPORT_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

panel_path = TABLE_DIR.parent / 'notebook03_phase_f' / 'phase_f_closed_loop_panel.parquet'
if not panel_path.exists():
    raise FileNotFoundError(f'Missing input: {panel_path}')

cols = [
    'patient_id', 'day', 'I_phase_b', 'I_stage_f_base', 'hazard_prob', 'hazard_prob_stage_f_base',
    'hazard_prob_stage_f', 'baseline_admission_event', 'stage_f_escalation_event',
    'cycle_id_stage_f', 'months_since_cycle_start', 'response_state_active'
]
df = pd.read_parquet(panel_path, columns=cols)

print('Rows:', len(df), 'Patients:', df['patient_id'].nunique())
df.head(3)

Rows: 4900000 Patients: 100000


,patient_id,day,I_phase_b,I_stage_f_base,hazard_prob,hazard_prob_stage_f_base,hazard_prob_stage_f,baseline_admission_event,stage_f_escalation_event,cycle_id_stage_f,months_since_cycle_start,response_state_active
0,P000000,0,0.399309,0.399309,0.003945,0.003945,0.003945,0,0,0,0,pre_admission
1,P000000,30,0.219145,0.219145,0.003946,0.003946,0.003946,0,0,0,1,pre_admission
2,P000000,60,0.120269,0.120269,0.003969,0.003969,0.003969,0,0,0,2,pre_admission


In [2]:
# Build observed/messy layer (realistic incompleteness, not random noise flood)
obs = df.copy()

# Proxy complexity from available columns
obs['complexity_proxy'] = (obs['I_stage_f_base'].fillna(0).astype(np.float32) + obs['cycle_id_stage_f'].astype(np.float32)).astype(np.float32)

# Missingness mechanisms (MNAR-like / MAR-like proxies)
p_missing_instability = np.clip(0.03 + 0.12 * (obs['response_state_active'].eq('nonresponse')).astype(float), 0.0, 0.35)
p_missing_hazard = np.clip(0.02 + 0.10 * (obs['months_since_cycle_start'] > 18).astype(float), 0.0, 0.25)
p_missing_cycle = np.clip(0.01 + 0.08 * (obs['baseline_admission_event'] == 0).astype(float), 0.0, 0.20)

u1 = np.random.rand(len(obs))
u2 = np.random.rand(len(obs))
u3 = np.random.rand(len(obs))

m1 = u1 < p_missing_instability
m2 = u2 < p_missing_hazard
m3 = u3 < p_missing_cycle

obs.loc[m1, 'I_stage_f_base'] = np.nan
obs.loc[m2, 'hazard_prob_stage_f'] = np.nan
obs.loc[m3, 'months_since_cycle_start'] = np.nan

for c in ['I_stage_f_base', 'hazard_prob_stage_f', 'months_since_cycle_start']:
    obs[f'{c}__is_missing'] = obs[c].isna().astype(np.int8)

missingness_summary = pd.DataFrame({
    'feature': ['I_stage_f_base', 'hazard_prob_stage_f', 'months_since_cycle_start'],
    'missing_rate': [
        float(obs['I_stage_f_base'].isna().mean()),
        float(obs['hazard_prob_stage_f'].isna().mean()),
        float(obs['months_since_cycle_start'].isna().mean())
    ],
    'missing_signal_nonresponse': [
        float(obs.loc[obs['response_state_active'].eq('nonresponse'), 'I_stage_f_base'].isna().mean()),
        float(obs.loc[obs['response_state_active'].eq('nonresponse'), 'hazard_prob_stage_f'].isna().mean()),
        float(obs.loc[obs['response_state_active'].eq('nonresponse'), 'months_since_cycle_start'].isna().mean())
    ]
})
missingness_summary.to_csv(TABLE_DIR / 'stage_g_missingness_summary.csv', index=False)

missingness_summary

,feature,missing_rate,missing_signal_nonresponse
0,I_stage_f_base,0.037230,0.149526
1,hazard_prob_stage_f,0.069125,0.048050
2,months_since_cycle_start,0.089114,0.086492


In [3]:
# Extreme-value storytelling (retain extremes, quantify their narrative value)
story = obs.copy()
story['I_stage_f_base_filled'] = story['I_stage_f_base'].fillna(story['I_stage_f_base'].median())
story['hazard_prob_stage_f_filled'] = story['hazard_prob_stage_f'].fillna(story['hazard_prob_stage_f'].median())

q_i = float(story['I_stage_f_base_filled'].quantile(0.995))
q_h = float(story['hazard_prob_stage_f_filled'].quantile(0.995))

story['extreme_instability'] = (story['I_stage_f_base_filled'] >= q_i).astype(np.int8)
story['extreme_hazard'] = (story['hazard_prob_stage_f_filled'] >= q_h).astype(np.int8)
story['extreme_any'] = ((story['extreme_instability'] + story['extreme_hazard']) > 0).astype(np.int8)

extreme_vs_non = pd.DataFrame([{
    'group': 'extreme_any=1',
    'n': int(story['extreme_any'].sum()),
    'event_rate': float(story.loc[story['extreme_any'] == 1, 'stage_f_escalation_event'].mean())
}, {
    'group': 'extreme_any=0',
    'n': int((story['extreme_any'] == 0).sum()),
    'event_rate': float(story.loc[story['extreme_any'] == 0, 'stage_f_escalation_event'].mean())
}])
extreme_vs_non['risk_ratio_vs_non_extreme'] = extreme_vs_non.loc[0, 'event_rate'] / max(extreme_vs_non.loc[1, 'event_rate'], 1e-9)
extreme_vs_non.to_csv(TABLE_DIR / 'stage_g_extreme_event_comparison.csv', index=False)

extreme_stories = story.loc[story['extreme_any'] == 1, [
    'patient_id', 'day', 'response_state_active', 'I_stage_f_base_filled', 'hazard_prob_stage_f_filled',
    'months_since_cycle_start', 'stage_f_escalation_event'
]].copy()
extreme_stories = extreme_stories.sort_values(['hazard_prob_stage_f_filled', 'I_stage_f_base_filled'], ascending=False).head(2000)
extreme_stories.to_csv(TABLE_DIR / 'stage_g_extreme_story_samples.csv', index=False)

plt.figure(figsize=(8,5))
plt.hist(story['hazard_prob_stage_f_filled'], bins=60)
plt.axvline(q_h, linestyle='--')
plt.title('Stage G Hazard Distribution with Extreme Threshold')
plt.xlabel('hazard_prob_stage_f')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_g_hazard_extreme_threshold.png', dpi=140, bbox_inches='tight')
plt.close()

extreme_vs_non

,group,n,event_rate,risk_ratio_vs_non_extreme
0,extreme_any=1,38771,0.080163,7.489156
1,extreme_any=0,4861229,0.010704,7.489156


In [4]:
# Stage-wise complexity ladder: S0 -> S1 -> S2
data = obs.copy()

# Keep data messy in raw layer; use model layer with imputation + missing flags
data['is_nonresponse'] = data['response_state_active'].eq('nonresponse').astype(np.int8)
data['is_partial'] = data['response_state_active'].eq('partial_response').astype(np.int8)
data['is_stabilized'] = data['response_state_active'].eq('stabilized').astype(np.int8)

data['I_stage_f_base_x_cycle'] = data['I_stage_f_base'].fillna(data['I_stage_f_base'].median()) * data['cycle_id_stage_f']
data['extreme_hazard_flag'] = (data['hazard_prob_stage_f'].fillna(data['hazard_prob_stage_f'].median()) >= data['hazard_prob_stage_f'].fillna(data['hazard_prob_stage_f'].median()).quantile(0.995)).astype(np.int8)

label = 'stage_f_escalation_event'

# Sample for training speed while preserving event ratio
pos = data[data[label] == 1]
neg = data[data[label] == 0].sample(n=min(len(data[data[label] == 0]), len(pos) * 3), random_state=42)
train_df = pd.concat([pos, neg], axis=0).sample(frac=1.0, random_state=42).reset_index(drop=True)

# Patient-wise split to reduce leakage
patients = train_df['patient_id'].drop_duplicates().sample(frac=1.0, random_state=42).to_numpy()
n = len(patients)
p_train = set(patients[: int(0.70 * n)])
p_val = set(patients[int(0.70 * n): int(0.85 * n)])
p_test = set(patients[int(0.85 * n):])

split = np.where(train_df['patient_id'].isin(p_train), 'train', np.where(train_df['patient_id'].isin(p_val), 'val', 'test'))
train_df['split'] = split

feature_sets = {
    'S0_base': ['cycle_id_stage_f', 'months_since_cycle_start'],
    'S1_plus_clinical': ['cycle_id_stage_f', 'months_since_cycle_start', 'I_stage_f_base', 'hazard_prob_stage_f', 'is_nonresponse', 'is_partial', 'is_stabilized', 'I_stage_f_base__is_missing', 'hazard_prob_stage_f__is_missing', 'months_since_cycle_start__is_missing'],
    'S2_plus_interactions_extremes': ['cycle_id_stage_f', 'months_since_cycle_start', 'I_stage_f_base', 'hazard_prob_stage_f', 'is_nonresponse', 'is_partial', 'is_stabilized', 'I_stage_f_base__is_missing', 'hazard_prob_stage_f__is_missing', 'months_since_cycle_start__is_missing', 'I_stage_f_base_x_cycle', 'extreme_hazard_flag']
}

def ece_score(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.digitize(y_prob, bins) - 1
    out = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.sum() == 0:
            continue
        out += abs(y_true[m].mean() - y_prob[m].mean()) * (m.sum() / len(y_true))
    return float(out)

rows = []
trained_models = {}

for stage_name, feats in feature_sets.items():
    X_train = train_df.loc[train_df['split'] == 'train', feats]
    y_train = train_df.loc[train_df['split'] == 'train', label].astype(int).to_numpy()
    X_test = train_df.loc[train_df['split'] == 'test', feats]
    y_test = train_df.loc[train_df['split'] == 'test', label].astype(int).to_numpy()

    num_cols = feats
    pre = ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols)], remainder='drop')

    lr = Pipeline([
        ('pre', pre),
        ('clf', LogisticRegression(penalty='elasticnet', l1_ratio=0.35, C=0.8, solver='saga', max_iter=600, n_jobs=-1))
    ])
    lr.fit(X_train, y_train)
    p_lr = lr.predict_proba(X_test)[:, 1]

    rf = Pipeline([
        ('pre', ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols)], remainder='drop')),
        ('clf', RandomForestClassifier(n_estimators=220, min_samples_leaf=8, random_state=42, n_jobs=-1))
    ])
    rf.fit(X_train, y_train)
    p_rf = rf.predict_proba(X_test)[:, 1]

    for model_name, p in [('elasticnet_logistic', p_lr), ('random_forest', p_rf)]:
        rows.append({
            'stage': stage_name,
            'model': model_name,
            'n_test': int(len(y_test)),
            'auroc': float(roc_auc_score(y_test, p)),
            'pr_auc': float(average_precision_score(y_test, p)),
            'brier': float(brier_score_loss(y_test, p)),
            'ece_10bin': ece_score(y_test, p, n_bins=10)
        })

    trained_models[(stage_name, 'elasticnet_logistic')] = (lr, feats, X_test.copy(), y_test.copy())

comparison = pd.DataFrame(rows).sort_values(['stage', 'model']).reset_index(drop=True)
comparison.to_csv(TABLE_DIR / 'stage_g_model_stage_comparison.csv', index=False)
comparison

c:\Users\Hope\miniforge3\envs\ml-ultra\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Hope\miniforge3\envs\ml-ultra\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\Hope\miniforge3\envs\ml-ultra\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='

,stage,model,n_test,auroc,pr_auc,brier,ece_10bin
0,S0_base,elasticnet_logistic,33194,0.990228,0.972283,0.023384,0.045813
1,S0_base,random_forest,33194,0.999858,0.999278,0.003322,0.000897
2,S1_plus_clinical,elasticnet_logistic,33194,0.999889,0.999494,0.003254,0.007053
3,S1_plus_clinical,random_forest,33194,0.999998,0.999995,0.000548,0.001658
4,S2_plus_interactions_extremes,elasticnet_logistic,33194,0.999883,0.999424,0.003273,0.007083
5,S2_plus_interactions_extremes,random_forest,33194,0.999998,0.999995,0.000554,0.002401


In [5]:
# Permutation diagnostics
# 1) Label permutation collapse check
stage_for_perm = 'S2_plus_interactions_extremes'
lr, feats, X_test_ref, y_test_ref = trained_models[(stage_for_perm, 'elasticnet_logistic')]

X_train_perm = train_df.loc[train_df['split'] == 'train', feats]
y_train_true = train_df.loc[train_df['split'] == 'train', 'stage_f_escalation_event'].astype(int).to_numpy()
y_train_shuf = np.random.permutation(y_train_true)

# retrain on shuffled labels
num_cols = feats
pre = ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols)], remainder='drop')
lr_shuf = Pipeline([
    ('pre', pre),
    ('clf', LogisticRegression(penalty='elasticnet', l1_ratio=0.35, C=0.8, solver='saga', max_iter=600, n_jobs=-1))
])
lr_shuf.fit(X_train_perm, y_train_shuf)

p_true = lr.predict_proba(X_test_ref)[:, 1]
p_shuf = lr_shuf.predict_proba(X_test_ref)[:, 1]

perm_collapse = pd.DataFrame([{
    'stage': stage_for_perm,
    'auroc_true_labels': float(roc_auc_score(y_test_ref, p_true)),
    'auroc_shuffled_labels': float(roc_auc_score(y_test_ref, p_shuf)),
    'pr_auc_true_labels': float(average_precision_score(y_test_ref, p_true)),
    'pr_auc_shuffled_labels': float(average_precision_score(y_test_ref, p_shuf))
}])
perm_collapse.to_csv(TABLE_DIR / 'stage_g_label_permutation_collapse.csv', index=False)

# 2) Feature permutation importance on test set
pi = permutation_importance(lr, X_test_ref, y_test_ref, n_repeats=8, random_state=42, scoring='roc_auc', n_jobs=-1)
perm_imp = pd.DataFrame({
    'feature': feats,
    'importance_mean': pi.importances_mean,
    'importance_std': pi.importances_std
}).sort_values('importance_mean', ascending=False)
perm_imp.to_csv(TABLE_DIR / 'stage_g_feature_permutation_importance.csv', index=False)

plt.figure(figsize=(9, 5))
top = perm_imp.head(12).iloc[::-1]
plt.barh(top['feature'], top['importance_mean'])
plt.title('Stage G Feature Permutation Importance (Top 12)')
plt.xlabel('Mean AUROC drop on permutation')
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_g_feature_permutation_importance_top12.png', dpi=140, bbox_inches='tight')
plt.close()

perm_collapse

c:\Users\Hope\miniforge3\envs\ml-ultra\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Hope\miniforge3\envs\ml-ultra\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


,stage,auroc_true_labels,auroc_shuffled_labels,pr_auc_true_labels,pr_auc_shuffled_labels
0,S2_plus_interactions_extremes,0.999883,0.447061,0.999424,0.263128


In [8]:
# Source-agnostic contract + adapter validation
CONTRACT_COLUMNS = [
    'patient_id', 'day', 'I_phase_b', 'I_stage_f_base', 'hazard_prob', 'hazard_prob_stage_f_base',
    'hazard_prob_stage_f', 'baseline_admission_event', 'stage_f_escalation_event',
    'cycle_id_stage_f', 'months_since_cycle_start', 'response_state_active'
 ]

CONTRACT_DTYPES = {
    'patient_id': 'object',
    'day': 'int',
    'I_phase_b': 'float',
    'I_stage_f_base': 'float',
    'hazard_prob': 'float',
    'hazard_prob_stage_f_base': 'float',
    'hazard_prob_stage_f': 'float',
    'baseline_admission_event': 'int',
    'stage_f_escalation_event': 'int',
    'cycle_id_stage_f': 'int',
    'months_since_cycle_start': 'float',
    'response_state_active': 'object'
}

def normalize_to_contract(df_in: pd.DataFrame) -> pd.DataFrame:
    out = df_in.copy()
    for col in CONTRACT_COLUMNS:
        if col not in out.columns:
            out[col] = np.nan

    out = out[CONTRACT_COLUMNS].copy()

    out['patient_id'] = out['patient_id'].astype('object')
    out['response_state_active'] = out['response_state_active'].astype('object')

    out['day'] = pd.to_numeric(out['day'], errors='coerce').fillna(0).astype(np.int32)
    out['baseline_admission_event'] = pd.to_numeric(out['baseline_admission_event'], errors='coerce').fillna(0).astype(np.int8)
    out['stage_f_escalation_event'] = pd.to_numeric(out['stage_f_escalation_event'], errors='coerce').fillna(0).astype(np.int8)
    out['cycle_id_stage_f'] = pd.to_numeric(out['cycle_id_stage_f'], errors='coerce').fillna(0).astype(np.int16)

    for c in ['I_phase_b', 'I_stage_f_base', 'hazard_prob', 'hazard_prob_stage_f_base', 'hazard_prob_stage_f', 'months_since_cycle_start']:
        out[c] = pd.to_numeric(out[c], errors='coerce').astype(np.float32)

    return out

def adapter_load(source: str = 'synthetic', registry_path: Path | None = None) -> pd.DataFrame:
    if source == 'synthetic':
        base = pd.read_parquet(panel_path, columns=CONTRACT_COLUMNS)
        return normalize_to_contract(base)
    if source == 'registry':
        if registry_path is None or (not registry_path.exists()):
            raise FileNotFoundError('Registry source requested but file not found.')
        reg = pd.read_parquet(registry_path) if registry_path.suffix.lower() == '.parquet' else pd.read_csv(registry_path)
        return normalize_to_contract(reg)
    raise ValueError("source must be 'synthetic' or 'registry'")

adapted_syn = adapter_load('synthetic')

contract_report = {
    'source': 'synthetic',
    'rows': int(len(adapted_syn)),
    'patients': int(adapted_syn['patient_id'].nunique()),
    'required_columns_present': bool(all(c in adapted_syn.columns for c in CONTRACT_COLUMNS)),
    'missing_rates': {c: float(adapted_syn[c].isna().mean()) for c in CONTRACT_COLUMNS},
    'event_rate': float(adapted_syn['stage_f_escalation_event'].mean())
}
with open(TABLE_DIR / 'stage_g_data_contract_report.json', 'w', encoding='utf-8') as f:
    json.dump(contract_report, f, indent=4)

# frozen holdout registry (for anti-retune governance)
freeze = {
    'split_strategy': 'patient-wise frozen split',
    'train_ratio': 0.70,
    'val_ratio': 0.15,
    'test_ratio': 0.15,
    'label': 'stage_f_escalation_event',
    'note': 'Do not alter split after reviewing model metrics.'
}
with open(TABLE_DIR / 'stage_g_holdout_freeze.json', 'w', encoding='utf-8') as f:
    json.dump(freeze, f, indent=4)

contract_report

{'source': 'synthetic',
 'rows': 4900000,
 'patients': 100000,
 'required_columns_present': True,
 'missing_rates': {'patient_id': 0.0,
  'day': 0.0,
  'I_phase_b': 0.0,
  'I_stage_f_base': 0.0,
  'hazard_prob': 0.0,
  'hazard_prob_stage_f_base': 0.0,
  'hazard_prob_stage_f': 0.0,
  'baseline_admission_event': 0.0,
  'stage_f_escalation_event': 0.0,
  'cycle_id_stage_f': 0.0,
  'months_since_cycle_start': 0.0,
  'response_state_active': 0.0},
 'event_rate': 0.011253469387755103}

## Stage G.1 — Source-Agnostic Data Contract & Adapter Guardrails

This block makes the pipeline defensible and reusable across synthetic and future registry data.

Rules enforced:
- Canonical schema contract for model pipeline inputs
- Source adapter (`synthetic` now, `registry` ready)
- Holdout freeze tracked separately from feature tweaking
- Adapter validation report saved as artifact

In [9]:
# Interpretation-first summary report
best_row = comparison.sort_values('auroc', ascending=False).iloc[0]
extreme_event = pd.read_csv(TABLE_DIR / 'stage_g_extreme_event_comparison.csv')
missing = pd.read_csv(TABLE_DIR / 'stage_g_missingness_summary.csv')
perm = pd.read_csv(TABLE_DIR / 'stage_g_label_permutation_collapse.csv').iloc[0]

with open(TABLE_DIR / 'stage_g_data_contract_report.json', 'r', encoding='utf-8') as f:
    contract_report = json.load(f)
with open(TABLE_DIR / 'stage_g_holdout_freeze.json', 'r', encoding='utf-8') as f:
    holdout_freeze = json.load(f)

lines = []
lines.append('Stage G Messy Data Storytelling Summary')
lines.append(f'rows_input: {len(df)}')
lines.append(f"patients_input: {df['patient_id'].nunique()}")
lines.append('')
lines.append('[Data Contract & Governance]')
lines.append(f"contract_source={contract_report['source']}")
lines.append(f"contract_required_columns_present={contract_report['required_columns_present']}")
lines.append(f"frozen_split_strategy={holdout_freeze['split_strategy']}")
lines.append(f"frozen_label={holdout_freeze['label']}")
lines.append('')
lines.append('[Missingness Story]')
for _, r in missing.iterrows():
    lines.append(f"{r['feature']}: missing_rate={r['missing_rate']:.4f}, nonresponse_missing_rate={r['missing_signal_nonresponse']:.4f}")
lines.append('')
lines.append('[Extreme Value Story]')
lines.append(f"extreme_event_rate={float(extreme_event.loc[extreme_event['group']=='extreme_any=1','event_rate'].iloc[0]):.6f}")
lines.append(f"non_extreme_event_rate={float(extreme_event.loc[extreme_event['group']=='extreme_any=0','event_rate'].iloc[0]):.6f}")
lines.append(f"risk_ratio_vs_non_extreme={float(extreme_event['risk_ratio_vs_non_extreme'].iloc[0]):.4f}")
lines.append('')
lines.append('[Model Ladder]')
lines.append(f"best_stage={best_row['stage']}")
lines.append(f"best_model={best_row['model']}")
lines.append(f"best_auroc={best_row['auroc']:.6f}")
lines.append(f"best_pr_auc={best_row['pr_auc']:.6f}")
lines.append('')
lines.append('[Permutation Diagnostic]')
lines.append(f"auroc_true_labels={float(perm['auroc_true_labels']):.6f}")
lines.append(f"auroc_shuffled_labels={float(perm['auroc_shuffled_labels']):.6f}")
lines.append(f"collapse_delta={float(perm['auroc_true_labels'] - perm['auroc_shuffled_labels']):.6f}")

with open(REPORT_DIR / 'stage_g_storytelling_summary.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

manifest = {
    'phase': 'G',
    'notebook': '04_stage_g_messy_data_storytelling.ipynb',
    'inputs': [
        'Results/tables/notebook03_phase_f/phase_f_closed_loop_panel.parquet'
    ],
    'outputs_tables': [
        'Results/tables/notebook04_stage_g/stage_g_missingness_summary.csv',
        'Results/tables/notebook04_stage_g/stage_g_extreme_event_comparison.csv',
        'Results/tables/notebook04_stage_g/stage_g_extreme_story_samples.csv',
        'Results/tables/notebook04_stage_g/stage_g_model_stage_comparison.csv',
        'Results/tables/notebook04_stage_g/stage_g_label_permutation_collapse.csv',
        'Results/tables/notebook04_stage_g/stage_g_feature_permutation_importance.csv',
        'Results/tables/notebook04_stage_g/stage_g_data_contract_report.json',
        'Results/tables/notebook04_stage_g/stage_g_holdout_freeze.json'
    ],
    'outputs_figures': [
        'Results/figures/notebook04_stage_g/stage_g_hazard_extreme_threshold.png',
        'Results/figures/notebook04_stage_g/stage_g_feature_permutation_importance_top12.png'
    ],
    'outputs_reports': [
        'Results/reports/notebook04_stage_g/stage_g_storytelling_summary.txt'
    ]
}
with open(META_DIR / 'phase_g_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=4)

proof = {
    'missingness_summary_generated': (TABLE_DIR / 'stage_g_missingness_summary.csv').exists(),
    'extreme_story_tables_generated': (TABLE_DIR / 'stage_g_extreme_story_samples.csv').exists(),
    'model_comparison_generated': (TABLE_DIR / 'stage_g_model_stage_comparison.csv').exists(),
    'label_permutation_collapse_generated': (TABLE_DIR / 'stage_g_label_permutation_collapse.csv').exists(),
    'feature_permutation_importance_generated': (TABLE_DIR / 'stage_g_feature_permutation_importance.csv').exists(),
    'data_contract_report_generated': (TABLE_DIR / 'stage_g_data_contract_report.json').exists(),
    'holdout_freeze_generated': (TABLE_DIR / 'stage_g_holdout_freeze.json').exists(),
    'summary_report_generated': (REPORT_DIR / 'stage_g_storytelling_summary.txt').exists(),
    'manifest_generated': (META_DIR / 'phase_g_manifest.json').exists()
}
with open(REPORT_DIR / 'stage_g_checklist_proof.json', 'w', encoding='utf-8') as f:
    json.dump({'proof': proof}, f, indent=4)

print('Stage G outputs generated')
for k, v in proof.items():
    print('-', k, ':', v)

comparison.sort_values('auroc', ascending=False).head(10)

Stage G outputs generated
- missingness_summary_generated : True
- extreme_story_tables_generated : True
- model_comparison_generated : True
- label_permutation_collapse_generated : True
- feature_permutation_importance_generated : True
- data_contract_report_generated : True
- holdout_freeze_generated : True
- summary_report_generated : True
- manifest_generated : True


,stage,model,n_test,auroc,pr_auc,brier,ece_10bin
5,S2_plus_interactions_extremes,random_forest,33194,0.999998,0.999995,0.000554,0.002401
3,S1_plus_clinical,random_forest,33194,0.999998,0.999995,0.000548,0.001658
2,S1_plus_clinical,elasticnet_logistic,33194,0.999889,0.999494,0.003254,0.007053
4,S2_plus_interactions_extremes,elasticnet_logistic,33194,0.999883,0.999424,0.003273,0.007083
1,S0_base,random_forest,33194,0.999858,0.999278,0.003322,0.000897
0,S0_base,elasticnet_logistic,33194,0.990228,0.972283,0.023384,0.045813


## Interpretation Notes You Asked For

- **Missingness**: modeled as mechanism-aware (not pure random deletion), with missingness indicators carried into modeling.
- **Permutation features**: implemented as **feature permutation importance** and **label permutation collapse** checks.
- **Messy real-world stance**: extremes are preserved, profiled, and compared against non-extreme outcomes, not dropped as outliers by default.
- **Story-first analysis**: every output is designed to answer "what does this data behavior mean clinically/operationally?"

In [ ]:
# Inline artifact gallery for this notebook stage
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = PROJECT_ROOT if 'PROJECT_ROOT' in globals() else (Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd())
STAGE_PREFIX = 'notebook04'

def _match_stage_dirs(base, prefix):
    if not base.exists():
        return []
    return sorted([p for p in base.glob(f'{prefix}*') if p.is_dir()])

def _show_table_file(path):
    suffix = path.suffix.lower()
    display(Markdown(f'**{path.name}**'))
    try:
        if suffix == '.csv':
            display(pd.read_csv(path).head(200))
        elif suffix == '.parquet':
            display(pd.read_parquet(path).head(200))
        elif suffix == '.json':
            data = json.loads(path.read_text(encoding='utf-8'))
            if isinstance(data, list):
                display(pd.DataFrame(data).head(200))
            elif isinstance(data, dict):
                display(pd.DataFrame([data]).T.head(200))
            else:
                print(str(data)[:12000])
        elif suffix in {'.txt', '.md'}:
            print(path.read_text(encoding='utf-8')[:12000])
    except Exception as exc:
        print(f'Could not render {path.name}: {exc}')

table_dirs = _match_stage_dirs(ROOT / 'Results' / 'tables', STAGE_PREFIX)
figure_dirs = _match_stage_dirs(ROOT / 'Results' / 'figures', STAGE_PREFIX)
report_dirs = _match_stage_dirs(ROOT / 'Results' / 'reports', STAGE_PREFIX)

display(Markdown(f'## Inline Artifact Gallery: {STAGE_PREFIX}'))
if not table_dirs and not figure_dirs and not report_dirs:
    print('No stage-matched artifact folders found yet. Run generation cells first.')

for d in table_dirs:
    display(Markdown(f'### Tables ({d.name})'))
    files = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.csv', '.parquet', '.json', '.txt'}])
    if not files:
        print('No table files found')
    for fp in files:
        _show_table_file(fp)

for d in figure_dirs:
    display(Markdown(f'### Visualizations ({d.name})'))
    imgs = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.png', '.jpg', '.jpeg'}])
    if not imgs:
        print('No figure files found')
    for fp in imgs:
        display(Markdown(f'**{fp.name}**'))
        display(Image(filename=str(fp)))

for d in report_dirs:
    display(Markdown(f'### Reports ({d.name})'))
    files = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.txt', '.md'}])
    if not files:
        print('No report files found')
    for fp in files:
        _show_table_file(fp)